# How do you control training with callbacks?
**Topics:** EarlyStopping · ModelCheckpoint · ReduceLROnPlateau · TensorBoard · Custom Callbacks

---
## 1. EarlyStopping

### What it is
- Stops training when a monitored metric stops improving
- Prevents overfitting and saves compute
- Optionally restores best weights when training stops

### Key arguments
- `monitor` — metric to watch: `'val_loss'`, `'val_accuracy'`
- `patience` — epochs to wait after last improvement before stopping
- `restore_best_weights=True` — reloads weights from best epoch automatically
- `min_delta` — minimum change to count as improvement
- `mode` — `'min'` for loss, `'max'` for accuracy (auto-detected)

### Gotchas
- Always monitor `val_loss` not `loss` — train loss always decreases
- `restore_best_weights=True` is critical — otherwise you get last epoch weights, not best
- Set `patience` generously — 5 to 20 epochs depending on how noisy val loss is

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42); tf.random.set_seed(42)

X = np.random.randn(1000, 8).astype(np.float32)
y = (X[:, 0] + X[:, 1] > 0).astype(np.int32)

model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(8,)),
    tf.keras.layers.Dense(64,  activation='relu'),
    tf.keras.layers.Dense(1,   activation='sigmoid'),
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True,
    verbose=1,
)

history = model.fit(
    X[:800], y[:800],
    epochs=100,
    batch_size=32,
    validation_data=(X[800:], y[800:]),
    callbacks=[early_stop],
    verbose=0,
)

stopped_epoch = early_stop.stopped_epoch
best_epoch    = stopped_epoch - early_stop.patience + 1
print(f"Stopped at epoch : {stopped_epoch + 1}")
print(f"Best epoch       : {best_epoch}")

epochs = range(1, len(history.history['loss'])+1)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs, history.history['loss'],     color='#3498DB', linewidth=2, label='Train loss')
ax.plot(epochs, history.history['val_loss'], color='#E74C3C', linewidth=2, label='Val loss', linestyle='--')
ax.axvline(best_epoch, color='#2ECC71', linewidth=2.5, linestyle='--',
           label=f'Best epoch ({best_epoch})')
ax.axvline(stopped_epoch+1, color='#F39C12', linewidth=2, linestyle=':',
           label=f'Stopped ({stopped_epoch+1})')
ax.fill_betweenx([0, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 1],
                  best_epoch, stopped_epoch+1, alpha=0.08, color='#E74C3C', label='Patience window')
ax.set_xlabel('Epoch', fontsize=12); ax.set_ylabel('Loss', fontsize=12)
ax.set_title('EarlyStopping — Training Halted Before Overfitting', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('early_stopping.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 2. ModelCheckpoint

### What it is
- Saves model weights (or full model) at specified intervals or when metric improves
- Ensures you never lose the best model even if training crashes

### Key arguments
- `filepath` — path to save model (can include `{epoch}` and `{val_loss:.2f}`)
- `monitor` — metric to watch
- `save_best_only=True` — only save when metric improves
- `save_weights_only=True` — save only weights, not full model
- `save_freq` — `'epoch'` or integer (save every N batches)

### Gotchas
- Use `save_best_only=True` to avoid filling disk with every-epoch checkpoints
- Combine with EarlyStopping — checkpoint saves best, EarlyStopping restores it
- File path with `{epoch:02d}` creates separate files per epoch

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os

np.random.seed(42); tf.random.set_seed(42)

X = np.random.randn(800, 8).astype(np.float32)
y = (X[:, 0] + X[:, 1] > 0).astype(np.int32)

model = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(8,)),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath='/tmp/best_model.weights.h5',
    monitor='val_loss',
    save_best_only=True,
    save_weights_only=True,
    verbose=1,
)

history = model.fit(
    X[:640], y[:640],
    epochs=30,
    batch_size=32,
    validation_data=(X[640:], y[640:]),
    callbacks=[checkpoint_cb],
    verbose=0,
)

# Load best weights
model.load_weights('/tmp/best_model.weights.h5')
results = model.evaluate(X[640:], y[640:], verbose=0)
print(f"Best model test accuracy: {results[1]:.3f}")

# Visualize which epochs were checkpointed
val_losses = history.history['val_loss']
best_val   = min(val_losses)
saved_epochs = [i+1 for i, v in enumerate(val_losses) if v <= best_val + 1e-6]

fig, ax = plt.subplots(figsize=(10, 5))
epochs = range(1, len(val_losses)+1)
ax.plot(epochs, val_losses, color='#E74C3C', linewidth=2, label='Val loss')
ax.scatter(saved_epochs, [val_losses[e-1] for e in saved_epochs],
           color='#2ECC71', s=80, zorder=5, label='Checkpoint saved', marker='*')
ax.set_xlabel('Epoch', fontsize=12); ax.set_ylabel('Val Loss', fontsize=12)
ax.set_title('ModelCheckpoint — Saves When Val Loss Improves', fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('model_checkpoint.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 3. ReduceLROnPlateau

### What it is
- Reduces learning rate when a metric stops improving
- Helps escape plateaus without a fixed schedule

### Key arguments
- `monitor` — metric to watch
- `factor` — LR multiplied by this when plateauing: `new_lr = lr * factor`
- `patience` — epochs to wait before reducing
- `min_lr` — minimum learning rate floor
- `cooldown` — epochs to wait after reduction before monitoring again

### Key intuition
- Large LR → fast initial progress → plateau → reduce LR → fine-tune to better minimum
- More adaptive than a fixed schedule — responds to actual training dynamics

### Gotchas
- `factor` should be < 1 (e.g. 0.5 or 0.1) — it's a multiplier not a subtraction
- Set `min_lr` to avoid LR becoming effectively zero
- Combine with EarlyStopping — ReduceLR first, then early stop

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42); tf.random.set_seed(42)

X = np.random.randn(1000, 8).astype(np.float32)
y = (X[:, 0] + X[:, 1] > 0).astype(np.int32)

model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(8,)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
              loss='binary_crossentropy', metrics=['accuracy'])

class LRTracker(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        self.model.history.history.setdefault('lr', []).append(
            float(self.model.optimizer.learning_rate)
        )

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1,
)
lr_tracker = LRTracker()

history = model.fit(
    X[:800], y[:800],
    epochs=60,
    batch_size=32,
    validation_data=(X[800:], y[800:]),
    callbacks=[reduce_lr, lr_tracker],
    verbose=0,
)

epochs = range(1, len(history.history['loss'])+1)
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

axes[0].plot(epochs, history.history['loss'],     color='#3498DB', linewidth=2, label='Train loss')
axes[0].plot(epochs, history.history['val_loss'], color='#E74C3C', linewidth=2, label='Val loss')
axes[0].set_ylabel('Loss', fontsize=11); axes[0].legend(fontsize=10)
axes[0].set_title('ReduceLROnPlateau — LR Reduces When Val Loss Plateaus', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

lrs = history.history.get('lr', [0.01]*len(epochs))
axes[1].plot(epochs, lrs, color='#9B59B6', linewidth=2.5)
axes[1].set_yscale('log')
axes[1].set_xlabel('Epoch', fontsize=11); axes[1].set_ylabel('Learning Rate (log)', fontsize=11)
axes[1].grid(True, alpha=0.3)
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('reduce_lr.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 4. TensorBoard Callback

### What it is
- Logs training metrics, model graph, histograms to TensorBoard
- Launch: `tensorboard --logdir=logs/`

### Key arguments
- `log_dir` — directory to write logs
- `histogram_freq` — how often to compute weight histograms (epochs)
- `write_graph` — whether to visualize model graph
- `update_freq` — `'epoch'` or integer (log every N batches)

### Gotchas
- Use different log_dir per run — include timestamp to avoid mixing runs
- `histogram_freq > 0` is expensive — use sparingly on large models
- Must pass validation data to see val metrics in TensorBoard

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import datetime

np.random.seed(42); tf.random.set_seed(42)

X = np.random.randn(800, 8).astype(np.float32)
y = (X[:, 0] + X[:, 1] > 0).astype(np.int32)

log_dir = '/tmp/logs/' + datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
tb_callback = tf.keras.callbacks.TensorBoard(
    log_dir=log_dir,
    histogram_freq=1,
    write_graph=True,
    update_freq='epoch',
)

model = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(8,)),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history = model.fit(X[:640], y[:640], epochs=20, batch_size=32,
                    validation_data=(X[640:], y[640:]),
                    callbacks=[tb_callback], verbose=0)

print(f"TensorBoard logs saved to: {log_dir}")
print("Launch with: tensorboard --logdir=/tmp/logs/")

epochs = range(1, 21)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(epochs, history.history['loss'],         color='#3498DB', linewidth=2, label='Train')
axes[0].plot(epochs, history.history['val_loss'],     color='#E74C3C', linewidth=2, label='Val', linestyle='--')
axes[0].set_title('Loss (logged to TensorBoard)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=10); axes[0].grid(True, alpha=0.3)
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

axes[1].plot(epochs, history.history['accuracy'],     color='#3498DB', linewidth=2, label='Train')
axes[1].plot(epochs, history.history['val_accuracy'], color='#E74C3C', linewidth=2, label='Val', linestyle='--')
axes[1].set_title('Accuracy (logged to TensorBoard)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.3)
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.suptitle('TensorBoard Callback Output', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('tensorboard_cb.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 5. Custom Callbacks

### What it is
- Subclass `tf.keras.callbacks.Callback` to add custom logic at any training event
- Override methods: `on_epoch_end`, `on_batch_end`, `on_train_begin`, etc.

### Common use cases
- Log custom metrics
- Implement custom learning rate schedules
- Send notifications (email, Slack) when training finishes
- Stop training based on custom conditions

### Gotchas
- `logs` dict contains current epoch metrics — keys match compile metrics
- `self.model` gives access to the model inside callback methods
- Return value is ignored — use `self.model.stop_training = True` to stop

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

X = np.random.randn(800, 8).astype(np.float32)
y = (X[:, 0] + X[:, 1] > 0).astype(np.int32)

class CustomLogger(tf.keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.epoch_logs = []

    def on_epoch_end(self, epoch, logs=None):
        lr = float(self.model.optimizer.learning_rate)
        self.epoch_logs.append({
            'epoch'    : epoch + 1,
            'loss'     : logs.get('loss', 0),
            'val_loss' : logs.get('val_loss', 0),
            'lr'       : lr,
        })
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}: loss={logs.get('loss',0):.4f}, val_loss={logs.get('val_loss',0):.4f}, lr={lr:.6f}")

    def on_train_end(self, logs=None):
        best = min(self.epoch_logs, key=lambda x: x['val_loss'])
        print(f"Best epoch: {best['epoch']} with val_loss={best['val_loss']:.4f}")

model = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(8,)),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

logger = CustomLogger()
all_cbs = [
    logger,
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=0),
]
history = model.fit(X[:640], y[:640], epochs=60, batch_size=32,
                    validation_data=(X[640:], y[640:]),
                    callbacks=all_cbs, verbose=0)

# Visualize callback interactions
epochs_logged = [d['epoch']    for d in logger.epoch_logs]
losses        = [d['loss']     for d in logger.epoch_logs]
val_losses    = [d['val_loss'] for d in logger.epoch_logs]
lrs           = [d['lr']       for d in logger.epoch_logs]

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(epochs_logged, losses,     color='#3498DB', linewidth=2, label='Train loss')
axes[0].plot(epochs_logged, val_losses, color='#E74C3C', linewidth=2, label='Val loss', linestyle='--')
axes[0].set_ylabel('Loss', fontsize=11); axes[0].legend(fontsize=10)
axes[0].set_title('Combined Callbacks: EarlyStopping + ReduceLROnPlateau + CustomLogger', fontsize=11, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

axes[1].plot(epochs_logged, lrs, color='#9B59B6', linewidth=2.5)
axes[1].set_yscale('log'); axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Learning Rate (log)', fontsize=11)
axes[1].grid(True, alpha=0.3)
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('combined_callbacks.png', dpi=120, bbox_inches='tight')
plt.show()


### Interview Q&A

**What callbacks would you use in a standard training pipeline?**
- Always: `EarlyStopping` + `ModelCheckpoint`
- Usually: `ReduceLROnPlateau` or a learning rate scheduler
- For debugging: `TensorBoard`

**How do you implement a warmup learning rate schedule?**
- Use `tf.keras.callbacks.LearningRateScheduler` with a custom function
- Or subclass `tf.keras.optimizers.schedules.LearningRateSchedule`

**What is the difference between `EarlyStopping` and `ReduceLROnPlateau`?**
- `ReduceLROnPlateau` reduces LR and lets training continue — tries to find a better minimum
- `EarlyStopping` stops training entirely — used as the final safety net
- Typical order: ReduceLR fires first (patience=5), EarlyStopping fires later (patience=15)

### Gotchas
- Order of callbacks in the list matters — they execute in order
- `restore_best_weights=True` in EarlyStopping makes `ModelCheckpoint` partially redundant but checkpoint is still useful for crash recovery
- Custom callbacks must call `super().__init__()` in `__init__`